# ⚽ Scratchformer — FIFA World Cup Training (Day 8 Iteration)

This notebook trains our from-scratch GPT model on the newly enhanced **FIFA World Cup** natural corpus (~2.3MB of Wikipedia history, iconic finals, legendary players, and rich match narratives) using a **Colab T4 GPU**.

**Key improvements in Day 8 to prevent overfitting and eliminate repetitive templates:**
1. **Natural Corpus 2.0:** 50+ rich Wikipedia articles (tournament histories from 1930 to 2026, famous finals, legends like Pelé, Maradona, Messi, Zidane, Ronaldo, Cruyff) + aggregated narrative match reports instead of isolated repetitive single-line templates.
2. **Dropout Regularization (`dropout=0.1`):** Active on embeddings, multi-head attention weights, residual projections, and feed-forward networks to prevent memorization.
3. **Tuned Training Schedule:** 5000 steps with AdamW, 300-step linear warmup, cosine LR decay, and weight decay (0.1).

---

### ⚡ Before you start
1. **Runtime → Change runtime type → T4 GPU**
2. Run all cells in order
3. Checkpoints are automatically saved to Google Drive under `scratchformer_checkpoints_fifa/`

## 1. Setup — Clone Repo & Install Dependencies

In [1]:
# Clone the latest code from GitHub
!rm -rf scratchformer
!git clone https://github.com/aryannten/scratchformer.git
%cd scratchformer
!pip install -q -r requirements.txt

Cloning into 'scratchformer'...
remote: Enumerating objects: 111, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (81/81), done.
remote: Total 111 (delta 51), reused 82 (delta 25), pack-reused 0 (from 0)
Receiving objects: 100% (111/111), 1.09 MiB | 4.98 MiB/s, done.
Resolving deltas: 100% (51/51), done.
/content/scratchformer


In [2]:
# Mount Google Drive for persistent checkpoint storage
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CHECKPOINT_DIR = '/content/drive/MyDrive/scratchformer_checkpoints_fifa'
import os
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)
print(f'Checkpoints will be saved to: {DRIVE_CHECKPOINT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints will be saved to: /content/drive/MyDrive/scratchformer_checkpoints_fifa


In [3]:
# Verify GPU is available
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:             {torch.cuda.get_device_name(0)}')
    total_mem = torch.cuda.mem_get_info(0)[1]
    print(f'Memory:          {total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected! Training will be very slow on CPU.')
    print('    Go to Runtime → Change runtime type → T4 GPU')

PyTorch version: 2.11.0+cu128
CUDA available:  True
GPU:             Tesla T4
Memory:          15.6 GB


## 2. Prepare the Natural FIFA World Cup Dataset 2.0

This fetches:
1. **50+ Wikipedia articles** covering all World Cup tournaments (1930–2026), famous finals, and legends (Pelé, Maradona, Messi, Zidane, Ronaldo, Cruyff, etc.).
2. **Aggregated match narrative reports** synthesizing matches, goals, stadiums, and awards into rich, varied paragraphs.

In [4]:
# Step 1: Fetch and assemble the natural FIFA corpus
# Creates data/raw/custom_corpus.txt (~2.3 MB)
!python fetch_custom_data.py

SCRATCHFORMER -- FIFA World Cup Dataset Builder 2.0

1. Fetching Wikipedia Articles (History, Tournaments, Legends)...
  [OK] Fetched 'FIFA World Cup' (38,909 chars)
  [OK] Fetched 'History of the FIFA World Cup' (35,960 chars)
  [OK] Fetched 'FIFA World Cup records and statistics' (22,159 chars)
  [OK] Fetched 'National team appearances in the FIFA World Cup' (4,115 chars)
  [OK] Fetched 'FIFA World Cup awards' (13,072 chars)
  [OK] Fetched '1930 FIFA World Cup' (22,352 chars)
  [OK] Fetched '1934 FIFA World Cup' (9,738 chars)
  [OK] Fetched '1938 FIFA World Cup' (9,327 chars)
  [OK] Fetched '1950 FIFA World Cup' (15,779 chars)
  [OK] Fetched '1954 FIFA World Cup' (14,722 chars)
  [OK] Fetched '1958 FIFA World Cup' (19,326 chars)
  [OK] Fetched '1962 FIFA World Cup' (15,863 chars)
  [OK] Fetched '1966 FIFA World Cup' (21,728 chars)
  [OK] Fetched '1970 FIFA World Cup' (23,087 chars)
  [OK] Fetched '1974 FIFA World Cup' (18,907 chars)
  [OK] Fetched '1978 FIFA World Cup' (31,397 chars)

In [5]:
# Step 2: Tokenize and split into train/val
# Creates:
#   data/prepared/custom_train.pt   — 90% train tokens
#   data/prepared/custom_val.pt     — 10% val tokens
#   data/prepared/custom_vocab.json — vocabulary mapping
!python prepare_data.py --dataset custom

Dataset 'custom' loaded: 2,417,378 characters.
Vocabulary size: 89 unique characters. Vocab saved to data/prepared/custom_vocab.json.
Train set has 2,175,640 tokens.
Validation set has 241,738 tokens.
Tensors saved successfully to data/prepared/custom_train.pt and data/prepared/custom_val.pt!


In [6]:
# Quick inspection of the prepared natural dataset
from tokenizer import CharTokenizer
import json

tokenizer = CharTokenizer.load('data/prepared/custom_vocab.json')
train_data = torch.load('data/prepared/custom_train.pt', weights_only=True)
val_data = torch.load('data/prepared/custom_val.pt', weights_only=True)

print(f'Vocab size:    {tokenizer.vocab_size} characters')
print(f'Train tokens:  {len(train_data):,}')
print(f'Val tokens:    {len(val_data):,}')
print(f'\nSample text (first 400 chars):')
print(tokenizer.decode(train_data[:400].tolist()))

Vocab size:    89 characters
Train tokens:  2,175,640
Val tokens:    241,738

Sample text (first 400 chars):
Wikipedia Templates+for+discussion Log 2025+December+5 Template Season+sidebar The FIFA World Cup is an international association football competition among the senior men's national teams of the members of the Federation Internationale de Football Association (FIFA), the sport's global governing body. The tournament has been held every four years since the inaugural tournament in 1930, with the e


## 3. Pre-Training Sanity Check

In [7]:
from model import Scratchformer, GPTConfig
from train import get_batch
import math

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Create model with dropout=0.1 for regularization
config = GPTConfig(vocab_size=tokenizer.vocab_size, dropout=0.1)
model = Scratchformer(config).to(device)

print(f'Model parameters: {model.count_parameters():,}')
print(f'Expected initial loss: {math.log(tokenizer.vocab_size):.4f}')

# Test forward pass
x, y = get_batch(train_data, batch_size=4, block_size=config.block_size, device=device)
logits, loss = model(x, y)

print(f'Output shape:    {logits.shape}  (expected: [4, {config.block_size}, {tokenizer.vocab_size}])')
print(f'Initial loss:    {loss.item():.4f}')
print(f'Loss is sane:    {"✅ Yes" if abs(loss.item() - math.log(tokenizer.vocab_size)) < 0.5 else "❌ No — investigate!"}')

del model
torch.cuda.empty_cache() if torch.cuda.is_available() else None

Model parameters: 831,065
Expected initial loss: 4.4886
Output shape:    torch.Size([4, 128, 89])  (expected: [4, 128, 89])
Initial loss:    4.7482
Loss is sane:    ✅ Yes


## 4. Train with Regularization! ⚽🚀

Training on the natural FIFA corpus for **5000 steps** with `dropout=0.1` and `weight_decay=0.1` on a T4 GPU (~5–7 minutes).

In [8]:
from train import train, TrainConfig
from model import GPTConfig

# ── Model architecture with dropout ───────────────────────
model_config = GPTConfig(
    block_size = 128,     # context window: 128 characters
    n_layer    = 4,       # 4 transformer blocks
    n_head     = 4,       # 4 attention heads
    n_embd     = 128,     # 128-dim hidden states
    dropout    = 0.1,     # 10% dropout to prevent overfitting
)

# ── Training hyperparameters ──────────────────────────────
train_config = TrainConfig(
    dataset        = 'custom',
    max_steps      = 5000,        # 5000 steps on the rich ~2.3MB corpus
    batch_size     = 64,          # standard batch size
    learning_rate  = 3e-4,        # AdamW LR
    weight_decay   = 0.1,         # decoupled weight decay
    grad_clip      = 1.0,         # gradient clipping
    warmup_steps   = 300,         # LR warmup
    eval_interval  = 250,         # evaluate every 250 steps
    save_interval  = 500,         # checkpoint every 500 steps
    checkpoint_dir = 'checkpoints',
)

# ── Launch training ───────────────────────────────────────
model, loss_log, tokenizer = train(
    model_config=model_config,
    train_config=train_config,
)

🧠 SCRATCHFORMER — Training
Device: cuda
GPU:    Tesla T4

Tokenizer: 89 characters
Model config: GPTConfig(vocab_size=89, block_size=128, n_layer=4, n_head=4, n_embd=128, dropout=0.1)

Loaded custom dataset:
  Train: 2,175,640 tokens
  Val:   241,738 tokens

Model parameters: 831,065 (0.83M)

Training for 5000 steps...
  Batch size: 64
  Block size: 128
  LR: 0.0003 (warmup 300 steps, cosine decay to 3e-05)
  Eval every 250 steps, save every 500 steps
------------------------------------------------------------


Training:   0%|                                                            | 0/5000 [00:01<?, ?it/s]

  Step     0 | Train loss: 4.6580 | Val loss: 4.6374 | LR: 1.00e-06 | Time: 1.3s
  💾 Checkpoint saved → checkpoints/best.pt


Training:   5%|█▎                       | 250/5000 [00:13<03:47, 20.91it/s, loss=2.6825, lr=2.5e-04]

  Step   250 | Train loss: 2.6591 | Val loss: 2.6218 | LR: 2.51e-04 | Time: 12.9s
  💾 Checkpoint saved → checkpoints/best.pt


...

  Step  4500 | Train loss: 1.4893 | Val loss: 0.9925 | LR: 3.75e-05 | Time: 207.5s


Training:  95%|██████████████████████▊ | 4753/5000 [03:38<00:41,  5.93it/s, loss=1.5656, lr=3.2e-05]

  Step  4750 | Train loss: 1.4741 | Val loss: 0.9822 | LR: 3.19e-05 | Time: 218.8s
  💾 Checkpoint saved → checkpoints/best.pt


Training: 100%|████████████████████████| 5000/5000 [03:50<00:00, 21.71it/s, loss=1.6186, lr=3.0e-05]


  Step  4999 | Train loss: 1.4732 | Val loss: 0.9847 | LR: 3.00e-05 | Time: 230.2s
  💾 Checkpoint saved → checkpoints/step_5000.pt
  💾 Checkpoint saved → checkpoints/final.pt
  📈 Loss curve saved → checkpoints/loss_curve.png

✅ TRAINING COMPLETE
  Total time:    231.9s (3.9 min)
  Final train:   1.4608
  Final val:     0.9832
  Best val:      0.9822
  Checkpoints:   checkpoints/
  Loss curve:    checkpoints/loss_curve.png



## 5. Results — Loss Curve

In [9]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(10, 6))

steps = [e['step'] for e in loss_log]
train_losses = [e['train'] for e in loss_log]
val_losses = [e['val'] for e in loss_log]

ax.plot(steps, train_losses, label='Train Loss', color='#4ECDC4', linewidth=2.5)
ax.plot(steps, val_losses, label='Val Loss', color='#FF6B6B', linewidth=2.5)
ax.set_xlabel('Step', fontsize=13)
ax.set_ylabel('Loss', fontsize=13)
ax.set_title('Scratchformer Training — Natural FIFA World Cup Corpus', fontsize=15, fontweight='bold')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)

if len(steps) > 0:
    ax.annotate(f'Train: {train_losses[-1]:.3f}', xy=(steps[-1], train_losses[-1]),
                fontsize=10, color='#4ECDC4', fontweight='bold')
    ax.annotate(f'Val: {val_losses[-1]:.3f}', xy=(steps[-1], val_losses[-1]),
                fontsize=10, color='#FF6B6B', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nFinal train loss: {train_losses[-1]:.4f}')
print(f'Final val loss:   {val_losses[-1]:.4f}')


Final train loss: 1.4608
Final val loss:   0.9832


## 6. Generate Natural Text — FIFA Football Stories ⚽

With the natural corpus and dropout regularization, generation produces diverse football prose, history, tactical descriptions, and narratives instead of rigid single-template lines.

In [10]:
def generate_text(model, tokenizer, prompt='', max_tokens=300, temperature=0.8, top_k=40):
    """Generate text from the trained model."""
    device = next(model.parameters()).device
    model.eval()

    if prompt:
        tokens = tokenizer.encode(prompt)
        idx = torch.tensor([tokens], dtype=torch.long, device=device)
    else:
        idx = torch.zeros((1, 1), dtype=torch.long, device=device)

    generated = model.generate(idx, max_new_tokens=max_tokens, temperature=temperature, top_k=top_k)
    return tokenizer.decode(generated[0].tolist())

In [11]:
# Test with varied natural prompts
prompts = [
    'The 1970 FIFA World Cup in Mexico',
    'Diego Maradona scored a memorable',
    'In the final match of the World Cup,',
    'Pele is widely considered',
    'The Brazilian national team',
    'Total Football was',
]

for prompt in prompts:
    print('=' * 60)
    print(f'📝 Prompt: "{prompt}"')
    print('=' * 60)
    output = generate_text(model, tokenizer, prompt=prompt, max_tokens=250, temperature=0.75, top_k=40)
    print(output)
    print()

📝 Prompt: "The 1970 FIFA World Cup in Mexico"
The 1970 FIFA World Cup in Mexico victory.

The 1994 FIFA World Cup, the finals World Cup had the national first historian to has decided his a champions he sommer and Two the ball Boinna, second by Sweded a goal for South Brazilian in the Sember Switzel Cruyff. As he greed football

📝 Prompt: "Diego Maradona scored a memorable"
Diego Maradona scored a memorable, he his first time and internaged the seasons. Brazil and that to the competitions. In the finals fans to the seconds a conviltion June Spaid Stadium in the Cruyff at Ding the quirteralified for the league winning a season, and him was the list of b

📝 Prompt: "In the final match of the World Cup,"
In the final match of the World Cup, and stage a crucial extra team. The world to and the game club and the first expoting best in 2014-06-26 saw to the 2018-18 in the ball-the FIFA Cup. On Messeppected a victory first Germany (in Group, while be the Italy conter-backi for Brazil goals

📝

In [12]:
print('=' * 60)
print('⚽ FREE GENERATION (Unconditioned, Temp=0.8)')
print('=' * 60)
for i in range(3):
    print(f'\n--- Story Sample {i+1} ---')
    print(generate_text(model, tokenizer, temperature=0.8, max_tokens=200))

⚽ FREE GENERATION (Unconditioned, Temp=0.8)

--- Story Sample 1 ---
	ulary suppectators in the 1975' minute, Reako fivised by Pall ammon, Argentina Barcelon within Eazian Mudradon in America Sainhon Argentina Campions all the Netherlands conred the first the 3-06 World

--- Story Sample 2 ---
	ded a 3-0 years. Rall rel camed as a for a most out-lic of the group stage on 18 Mexico 1956. In 1986-06-04 after canted a first met, for Spaire in the 1938 FIFA Men's Witherld Cup, Barcelia Madrid Cr

--- Story Sample 3 ---
	 the 1996 FIFA Men's World Cup, his cullained to the winner team contramer met from the mates of players in the pressivisional lime of the World Cup when Dankon, and diffended a first by Hugonal (Mara


## 7. Copy Checkpoints to Google Drive

Save `best.pt`, `final.pt`, and `custom_vocab.json` to Google Drive.

In [13]:
import shutil

for ckpt_name in ['best.pt', 'final.pt', 'loss_curve.png']:
    src = f'checkpoints/{ckpt_name}'
    dst = f'{DRIVE_CHECKPOINT_DIR}/{ckpt_name}'
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'✅ Copied {ckpt_name} → {dst}')
    else:
        print(f'⚠️  {src} not found, skipping')

shutil.copy2('data/prepared/custom_vocab.json', f'{DRIVE_CHECKPOINT_DIR}/custom_vocab.json')
print(f'✅ Copied custom_vocab.json → {DRIVE_CHECKPOINT_DIR}/custom_vocab.json')

print(f'\n📁 Drive contents:')
for f in os.listdir(DRIVE_CHECKPOINT_DIR):
    size = os.path.getsize(os.path.join(DRIVE_CHECKPOINT_DIR, f))
    print(f'   {f:30s} {size / 1e6:.1f} MB')

✅ Copied best.pt → /content/drive/MyDrive/scratchformer_checkpoints_fifa/best.pt
✅ Copied final.pt → /content/drive/MyDrive/scratchformer_checkpoints_fifa/final.pt
✅ Copied loss_curve.png → /content/drive/MyDrive/scratchformer_checkpoints_fifa/loss_curve.png
✅ Copied custom_vocab.json → /content/drive/MyDrive/scratchformer_checkpoints_fifa/custom_vocab.json

📁 Drive contents:
   best.pt                        11.1 MB
   final.pt                       11.1 MB
   loss_curve.png                 0.1 MB
   custom_vocab.json              0.0 MB
